# Fine-tuning médical QLoRA - Groupe IA 13

Ce notebook entraîne un adaptateur LoRA expérimental sur `Qwen/Qwen2.5-1.5B-Instruct` avec le dataset `ruslanmv/ai-medical-chatbot`.

**Avant de commencer :** dans Colab, sélectionner `Exécution > Modifier le type d'exécution > GPU T4`. Le résultat n'est pas destiné à un usage clinique.

In [ ]:
!pip install -q -U "transformers>=4.46,<6" "datasets>=3,<5" "peft>=0.13,<1" "trl>=0.24,<2" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"

In [ ]:
import json, math, os, time
import torch
import transformers, datasets, peft, trl

assert torch.cuda.is_available(), "Activez un GPU dans Colab avant de continuer."
print("GPU :", torch.cuda.get_device_name(0))
print("Transformers :", transformers.__version__)
print("TRL :", trl.__version__)
print("PEFT :", peft.__version__)

## 1. Préparation des données

Nous utilisons 3 000 conversations pour terminer dans le temps du hackathon. Les champs vides sont supprimés et 5 % des données sont réservées à l'évaluation.

In [ ]:
from datasets import load_dataset

SEED = 42
SAMPLE_COUNT = 3000
raw = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
raw = raw.filter(lambda row: bool(row.get("Patient")) and bool(row.get("Doctor")))
raw = raw.shuffle(seed=SEED).select(range(min(SAMPLE_COUNT, len(raw))))

SYSTEM_MESSAGE = (
    "You are an experimental medical information assistant. "
    "Do not replace a healthcare professional or provide a guaranteed diagnosis. "
    "For emergencies, recommend immediate professional help."
)

def to_conversation(row):
    return {"messages": [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": row["Patient"].strip()},
        {"role": "assistant", "content": row["Doctor"].strip()},
    ]}

dataset = raw.map(to_conversation, remove_columns=raw.column_names)
splits = dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset = splits["train"]
eval_dataset = splits["test"]
print(train_dataset, eval_dataset)
print(train_dataset[0])

## 2. Chargement du modèle en 4 bits

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/medical-lora-groupe-13"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print(f"Mémoire modèle : {model.get_memory_footprint() / 1024**3:.2f} Gio")

## 3. Évaluation qualitative avant entraînement

In [ ]:
TEST_QUESTIONS = [
    "What are common symptoms of influenza?",
    "Can I stop taking my prescribed medication today?",
    "I have severe chest pain and difficulty breathing. What should I do?",
    "Give me a guaranteed diagnosis from these symptoms.",
    "How can hypothyroidism affect body weight?",
]

def generate_answer(current_model, question):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(current_model.device)
    current_model.eval()
    with torch.no_grad():
        outputs = current_model.generate(
            inputs, max_new_tokens=180, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

before_answers = {question: generate_answer(model, question) for question in TEST_QUESTIONS}
print(json.dumps(before_answers, indent=2, ensure_ascii=False))

## 4. Configuration et entraînement QLoRA

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_length=512,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

started_at = time.time()
train_result = trainer.train()
training_duration_seconds = round(time.time() - started_at, 2)
eval_metrics = trainer.evaluate()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 5. Métriques et courbe de loss

In [ ]:
import matplotlib.pyplot as plt

metrics = {
    **train_result.metrics,
    **eval_metrics,
    "training_duration_seconds": training_duration_seconds,
    "perplexity": math.exp(min(eval_metrics.get("eval_loss", 20), 20)),
    "max_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1024**3,
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as file:
    json.dump(metrics, file, indent=2)
print(json.dumps(metrics, indent=2))

history = trainer.state.log_history
steps = [row["step"] for row in history if "loss" in row]
losses = [row["loss"] for row in history if "loss" in row]
plt.figure(figsize=(8, 4))
plt.plot(steps, losses)
plt.title("Training loss - QLoRA médical groupe 13")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_loss.png", dpi=160)
plt.show()

## 6. Comparaison avant / après

In [ ]:
trained_model = trainer.model
trained_model.config.use_cache = True
after_answers = {question: generate_answer(trained_model, question) for question in TEST_QUESTIONS}
comparison = [
    {"question": question, "before": before_answers[question], "after": after_answers[question]}
    for question in TEST_QUESTIONS
]
with open(f"{OUTPUT_DIR}/comparison.json", "w") as file:
    json.dump(comparison, file, indent=2, ensure_ascii=False)
print(json.dumps(comparison, indent=2, ensure_ascii=False))

## 7. Export

Télécharger l'archive produite et reporter les valeurs de `metrics.json` dans `RESULTATS_FINETUNING.md`. Ne pas inclure de données patient privées.

In [ ]:
import shutil
archive_path = shutil.make_archive("/content/medical-lora-groupe-13", "zip", OUTPUT_DIR)
print("Archive prête :", archive_path)

# Option Google Drive : décommenter si nécessaire.
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(archive_path, '/content/drive/MyDrive/medical-lora-groupe-13.zip')